# Deep Learning 004 — The Perceptron

The whole object is `step(w·x + b)`. This notebook builds that, draws the line it
implies, and then moves the bias around until what the bias *does* is obvious.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/placement.csv')
X = df[['cgpa', 'resume_score']].to_numpy()
y = df['placed'].to_numpy().astype(int)
print(X.shape, y.shape, '| balance', np.bincount(y))

## The forward pass

Two lines. `z` is a weighted sum plus an offset; `step` turns it into a decision.

In [ ]:
def step(z):
    return (z >= 0).astype(int)

def perceptron(X, w, b):
    return step(X @ w + b)

w = np.array([1.0, 1.0])
b = -10.0
pred = perceptron(X, w, b)
print('predictions (first 10):', pred[:10])
print('accuracy with a guessed w, b:', (pred == y).mean())

## The line it implies

`w·x + b = 0` is a straight line. Solve for `x2`:

$$w_1x_1 + w_2x_2 + b = 0 \;\Longrightarrow\; x_2 = -\frac{w_1}{w_2}x_1 - \frac{b}{w_2}$$

So **`w` sets the slope and `b` sets the intercept** — that is the entire geometry.

In [ ]:
def plot_boundary(w, b, ax, label=None, **kw):
    xs = np.linspace(X[:, 0].min() - .5, X[:, 0].max() + .5, 100)
    ax.plot(xs, -(w[0] / w[1]) * xs - b / w[1], label=label, **kw)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X[y == 0, 0], X[y == 0, 1], marker='o', label='not placed')
ax.scatter(X[y == 1, 0], X[y == 1, 1], marker='^', label='placed')
plot_boundary(w, b, ax, label=f'w={w}, b={b}', color='k')
ax.set(xlabel='cgpa', ylabel='resume_score',
       ylim=(X[:, 1].min() - .5, X[:, 1].max() + .5))
ax.legend(); plt.tight_layout(); plt.show()

## What the bias does

Hold `w` fixed and sweep `b`. The line **translates without rotating** — and the
accuracy traces out a curve with a clear best value.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].scatter(X[y == 0, 0], X[y == 0, 1], marker='o', s=18)
ax[0].scatter(X[y == 1, 0], X[y == 1, 1], marker='^', s=18)

biases = np.linspace(-16, -4, 7)
for bb in biases:
    plot_boundary(w, bb, ax[0], label=f'b={bb:.0f}', lw=1)
ax[0].set(title='same w, seven biases: parallel lines',
          ylim=(X[:, 1].min() - .5, X[:, 1].max() + .5))
ax[0].legend(fontsize=7)

sweep = np.linspace(-20, 0, 200)
acc = [(perceptron(X, w, bb) == y).mean() for bb in sweep]
ax[1].plot(sweep, acc)
best = sweep[int(np.argmax(acc))]
ax[1].axvline(best, color='crimson', ls='--',
              label=f'best b = {best:.2f} -> {max(acc):.1%}')
ax[1].set(xlabel='b', ylabel='accuracy', title='accuracy as the line slides')
ax[1].legend(); plt.tight_layout(); plt.show()
print(f'best bias {best:.2f} gives accuracy {max(acc):.1%}')

> **Without a bias the line must pass through the origin.** Set `b = 0` and re-run the
> sweep — you lose the ability to place the boundary anywhere except through `(0, 0)`,
> and the best achievable accuracy drops. That is the whole argument for having a bias
> term, in one experiment.

In [ ]:
# Rotation, for contrast: change w and the line pivots.
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X[y == 0, 0], X[y == 0, 1], marker='o', s=18)
ax.scatter(X[y == 1, 0], X[y == 1, 1], marker='^', s=18)
for angle in np.linspace(0.2, 1.4, 5):
    ww = np.array([np.cos(angle), np.sin(angle)])
    plot_boundary(ww, best, ax, label=f'angle {angle:.1f}', lw=1)
ax.set(title='changing w rotates the line',
       ylim=(X[:, 1].min() - .5, X[:, 1].max() + .5))
ax.legend(fontsize=7); plt.tight_layout(); plt.show()

## Exercises

1. Grid-search `w` (as an angle) **and** `b` together. What is the best accuracy any
   single line achieves on this data? Is it 100%?
2. Replace `step` with `sigmoid`. The boundary is in the same place — what changed?
3. Scale `w` and `b` by 10 simultaneously. Predictions are identical. Why? What does
   that tell you about the perceptron's parameters being non-unique?